<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/exams/final/FIN3_funding_panel_policies.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Managing Research Projects

## Project 3: Funding panel policies: Ranking, partial lotteries and bias

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

A call with 100 proposals and funds for 20 is decided by noisy reviewers who slightly favour established applicants. The example compares pure ranking, a partial lottery that funds the clear leaders and draws the rest from the next band, and a lottery above a threshold, with respect to the quality funded and the share of established applicants [1, 2].

**Required extensions.** Add the costs of reviewing, panels of different sizes, and bias that grows with the reviewers' uncertainty. Identify the conditions under which partial lotteries lose little quality while reducing bias, and discuss the implications for applicants and funders, including the programmes of Weeks 4 and 5.

## What this example implements

The notebook simulates 500 calls per policy, reports the mean true quality of funded proposals, the share of funded proposals among the truly best and the share of established applicants, and plots how reviewer noise affects ranking and partial lotteries.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Three decision policies

In [ ]:
import pandas as pd
rng = np.random.default_rng(SEED)

def run_call(policy, n=100, budget=20, reviewers=3, sd=0.8, bias=0.3):
    quality = rng.normal(3.5, 0.6, n)
    established = rng.random(n) < 0.4
    mean = (quality[:, None] + bias * established[:, None] + rng.normal(0, sd, (n, reviewers))).mean(1)
    order = np.argsort(-mean)
    if policy == "ranking":
        funded = order[:budget]
    elif policy == "partial lottery":
        funded = np.r_[order[:10], rng.choice(order[10:40], budget - 10, replace=False)]
    else:
        funded = rng.choice(np.where(mean >= np.quantile(mean, 0.6))[0], budget, replace=False)
    best = set(np.argsort(-quality)[:budget])
    return dict(mean_quality_funded=quality[funded].mean(), share_truly_best=len(best & set(funded)) / budget,
                established_share=established[funded].mean(), established_population=established.mean())

rows = []
for policy in ["ranking", "partial lottery", "threshold lottery"]:
    res = pd.DataFrame([run_call(policy) for _ in range(500)]).mean()
    rows.append(dict(policy=policy, **res))
pd.DataFrame(rows).set_index("policy").round(3)

## 2. Sensitivity to reviewer noise

In [ ]:
grid = [0.2, 0.5, 0.8, 1.2, 1.6]
for policy in ["ranking", "partial lottery"]:
    q = [pd.DataFrame([run_call(policy, sd=s) for _ in range(300)])["share_truly_best"].mean() for s in grid]
    plt.plot(grid, q, "o-", label=policy)
plt.xlabel("reviewer noise (standard deviation)"); plt.ylabel("share of funded proposals among the truly best"); plt.legend(); plt.show()

## Report and submission

Both tracks follow the course report template. The report states a question, justifies the approach, presents evidence from the literature or from the simulation with figures, discusses limitations and ends with recommendations for managing research projects. Sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Pier, E. L., Brauer, M., Filut, A., Kaatz, A., Raclaw, J., Nathan, M. J., Ford, C. E., & Carnes, M. (2018). Low agreement among reviewers evaluating the same NIH grant applications. *Proceedings of the National Academy of Sciences*, 115(12), 2952-2957. <https://doi.org/10.1073/pnas.1714379115>

[2] Graves, N., Barnett, A. G., & Clarke, P. (2011). Funding grant proposals for scientific research: Retrospective analysis of scores by members of grant review panel. *BMJ*, 343, d4797. <https://doi.org/10.1136/bmj.d4797>